# PPO Pairs Trading Agents

In [ ]:
import sys, os
sys.path.insert(0, '.')   # make env importable

import numpy as np
from stable_baselines3 import PPO
from env import PairsTradingEnv

import warnings; warnings.filterwarnings('ignore')


PAIR_NAME        = 'V_MA'  # ticker pair to evaluate
DATA_PATH        = f'./regime_output_{PAIR_NAME}/generated.npz'
TRANSACTION_COST = 0.001
ZSCORE_WINDOW    = 20
TOTAL_TIMESTEPS  = 250_000 
SEED             = 42   # data split + RNG
SEEDS            = [7, 8, 12, 15, 42, 43, 67, 123, 456, 789]  # training seeds
N_ENVS           = 4
CALIB_ALPHA      = 0.05  # slower EMA for real data: equilibria drift over years not days

np.random.seed(SEED)

In [ ]:
import yfinance as yf

TICKER_A   = 'V'
TICKER_B   = 'MA'
START_DATE = '2009-01-01'
END_DATE   = '2022-12-31'   # training end (test_evaluation starts 2023-01-01)

raw = yf.download([TICKER_A, TICKER_B], start=START_DATE, end=END_DATE,
                  auto_adjust=True, progress=False)['Close'].dropna()
print(f'{len(raw)} rows  ({raw.index[0].date()} -> {raw.index[-1].date()})')

log_A = np.log(raw[TICKER_A].values).astype(np.float64)
log_B = np.log(raw[TICKER_B].values).astype(np.float64)
assert len(log_A) == len(log_B)

## 1. Load Generated Data

In [ ]:
import joblib
from scipy.stats import multivariate_normal as _mvn
from statsmodels.tools import add_constant
from statsmodels.regression.linear_model import OLS
from numpy.linalg import eig as _eig

T_hist = len(log_A)
HORIZON     = 252   # episode length
STRIDE      = 5    # sliding window stride in days
BETA_WINDOW = 252   # rolling window for per-episode beta estimation


hmm_model  = joblib.load(f'./regime_output_{PAIR_NAME}/hmm_model.pkl')
scaler     = np.load(f'./regime_output_{PAIR_NAME}/feat_scaler.npy')
feat_means = scaler[0]
feat_stds  = scaler[1]


beta_global   = float(OLS(log_A, add_constant(log_B)).fit().params[1])
spread_global = log_A - beta_global * log_B

def compute_feat(spread, t, window=20, rw=60, th=1.0):
    lo = max(0, t - window)
    w  = spread[lo:t+1]
    if len(w) < 3:
        return None
    z  = float((w[-1] - w.mean()) / (w.std() + 1e-8))
    rv = np.diff(w).std() * np.sqrt(252) if len(w) > 1 else 1e-6
    lv = float(np.log(max(rv, 1e-6)))
    lo_r = max(0, t - rw)
    w_r  = spread[lo_r:t+1]
    if len(w_r) < 10:
        rate = 0.5
    else:
        mu_   = w_r.mean(); std_ = w_r.std() + 1e-8
        zs    = (w_r[:-1] - mu_) / std_
        exc   = np.abs(zs) > th
        rev   = (np.diff(w_r) * (mu_ - w_r[:-1])) > 0
        ns    = exc.sum()
        rate  = float((exc & rev).sum() / ns) if ns > 0 else 0.5
    lr = float(np.log(np.clip(rate, 0.05, 0.95) /
                      (1 - np.clip(rate, 0.05, 0.95))))
    return np.array([z, lv, lr], dtype=np.float32)

# Causal HMM forward filter
ev_, vc = _eig(hmm_model.transmat_.T)
st_     = np.real(vc[:, np.argmin(np.abs(ev_ - 1.0))])
stationary = (st_ / st_.sum()).astype(np.float32)

def filt_step(alpha, feat_sc):
    ap  = hmm_model.transmat_.T @ alpha
    ll  = np.array([_mvn.logpdf(feat_sc,
                                mean=hmm_model.means_[k],
                                cov=hmm_model.covars_[k])
                    for k in range(3)])
    ll -= ll.max(); lk = np.exp(ll); an = ap * lk
    tot = an.sum()
    return (an / tot).astype(np.float32) if tot > 1e-12 else alpha

alpha       = stationary.copy()
regime_hard = np.zeros(T_hist, dtype=np.int8)
print('Running causal HMM forward filter...')
for t in range(T_hist):
    fr = compute_feat(spread_global, t)
    if fr is not None:
        alpha = filt_step(alpha, (fr - feat_means) / (feat_stds + 1e-8))
    regime_hard[t] = int(np.argmax(alpha))

freq = {r: np.mean(regime_hard == r) for r in range(3)}
print(f'Regime frequencies — stable: {freq[0]:.1%}  '
      f'neutral: {freq[1]:.1%}  crisis: {freq[2]:.1%}')

episodes_sp = []
episodes_rg = []
beta_list   = []

for start in range(0, T_hist - HORIZON, STRIDE):
    end     = start + HORIZON
    lo_beta = max(0, start - BETA_WINDOW)
    if start - lo_beta < 60:
        continue   # skip if not enough data for reliable beta
    beta_ep     = float(OLS(log_A[lo_beta:start],
                            add_constant(log_B[lo_beta:start])
                            ).fit().params[1])
    spread_ep   = (log_A[start:end] - beta_ep * log_B[start:end]
                   ).astype(np.float32)
    episodes_sp.append(spread_ep)
    episodes_rg.append(regime_hard[start:end])
    beta_list.append(beta_ep)

spreads = np.array(episodes_sp, dtype=np.float32)
regimes = np.array(episodes_rg, dtype=np.int8)
N, T    = spreads.shape

# Use median rolling beta for dollar P&L scaling
BETA = float(np.median(beta_list))
os.makedirs(f'models_{PAIR_NAME}_real', exist_ok=True)
np.save(f'models_{PAIR_NAME}_real/beta.npy', np.array([BETA]))

print(f'Created {N} episodes of {T} steps  (stride={STRIDE})')
print(f'Median rolling beta: {BETA:.4f}')

n_train = int(0.8 * N)
sp_tr, rg_tr = spreads[:n_train], regimes[:n_train]
sp_ev, rg_ev = spreads[n_train:], regimes[n_train:]
n_eval = len(sp_ev)

print(f'Train: {n_train} episodes  |  Eval: {n_eval} episodes')
print(f'Spread range — mean={spreads.mean():.4f}  std={spreads.std():.4f}')

## 2. Train Baseline Agent (No Regimes)

In [ ]:
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import (
    EvalCallback, StopTrainingOnNoModelImprovement
)
from stable_baselines3.common.monitor import Monitor
import torch

HYPERPARAMS = dict(
    learning_rate = 3e-4,
    n_steps       = T,
    batch_size    = 64,
    n_epochs      = 10,
    gamma         = 0.99,
    gae_lambda    = 0.95,
    clip_range    = 0.2,
    ent_coef      = 0.005,
    vf_coef       = 0.5,
    max_grad_norm = 0.5,
    policy_kwargs = dict(
        net_arch      = [128, 128],
        activation_fn = torch.nn.Tanh,
        log_std_init  = -1.0,
    ),
)


import pickle as _pickle
_ou = _pickle.load(open(f'./regime_output_{PAIR_NAME}/ou_params.pkl', 'rb'))
REG_EXTRA = dict(mu_r=_ou['mu_r'], sigma_r=_ou['sigma_r'], kappa=_ou['kappa'])

def make_env(sp, rg, use_reg):
    def _fn():
        extra = REG_EXTRA if use_reg else {}
        return Monitor(PairsTradingEnv(
            sp, rg,
            use_regimes      = use_reg,
            transaction_cost = TRANSACTION_COST,
            zscore_window    = ZSCORE_WINDOW,
            calib_alpha      = CALIB_ALPHA,
            **extra,
        ))
    return _fn

eval_env_base = Monitor(PairsTradingEnv(
    sp_ev, rg_ev, use_regimes=False,
    transaction_cost=TRANSACTION_COST, zscore_window=ZSCORE_WINDOW
))


for s in SEEDS:
    save_dir = f'models_{PAIR_NAME}_real/baseline/seed{s}'
    log_dir  = f'logs_{PAIR_NAME}_real/baseline/seed{s}'
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir,  exist_ok=True)

    train_env_base = make_vec_env(
        make_env(sp_tr, rg_tr, False), n_envs=N_ENVS, seed=s
    )

    stop_cb_base = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals = 10,   # stop after 10 evals with no improvement
        min_evals                = 20,   # always run at least 20 evals first
        verbose                  = 1,
    )
    eval_cb_base = EvalCallback(
        eval_env_base,
        best_model_save_path = f'{save_dir}/',
        log_path             = f'{log_dir}/',
        eval_freq            = TOTAL_TIMESTEPS // 50,
        n_eval_episodes      = min(50, n_eval),
        deterministic        = True,
        callback_after_eval  = stop_cb_base,
        verbose              = 0,
    )

    model_base = PPO('MlpPolicy', train_env_base,
                     **HYPERPARAMS, verbose=0, seed=s)
    print(f'Training baseline agent (seed={s})...')
    model_base.learn(TOTAL_TIMESTEPS, callback=eval_cb_base, progress_bar=True)
    model_base.save(f'{save_dir}/final')

print(f'Baseline training complete for seeds {SEEDS}.')

## 3. Train Regime-Aware Agent

In [ ]:
eval_env_reg = Monitor(PairsTradingEnv(
    sp_ev, rg_ev, use_regimes=True,
    transaction_cost=TRANSACTION_COST, zscore_window=ZSCORE_WINDOW,
    calib_alpha=CALIB_ALPHA,
    **REG_EXTRA
))

HYPERPARAMS_REGIME = dict(
    learning_rate = 3e-4,
    n_steps       = T,
    batch_size    = 64,
    n_epochs      = 10,
    gamma         = 0.99,
    gae_lambda    = 0.95,
    clip_range    = 0.2,
    ent_coef      = 0.005,
    vf_coef       = 0.5,
    max_grad_norm = 0.5,
    policy_kwargs = dict(
        net_arch      = [128, 128],
        activation_fn = torch.nn.Tanh,
        log_std_init  = -1.0,
    ),
)
TOTAL_TIMESTEPS_REGIME = 250_000

for s in SEEDS:
    save_dir = f'models_{PAIR_NAME}_real/regime/seed{s}'
    log_dir  = f'logs_{PAIR_NAME}_real/regime/seed{s}'
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir,  exist_ok=True)

    train_env_reg = make_vec_env(
        make_env(sp_tr, rg_tr, True), n_envs=N_ENVS, seed=s
    )

    stop_cb_reg = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals = 10,
        min_evals                = 20,
        verbose                  = 1,
    )
    eval_cb_reg = EvalCallback(
        eval_env_reg,
        best_model_save_path = f'{save_dir}/',
        log_path             = f'{log_dir}/',
        eval_freq            = TOTAL_TIMESTEPS_REGIME // 50,
        n_eval_episodes      = min(50, n_eval),
        deterministic        = True,
        callback_after_eval  = stop_cb_reg,
        verbose              = 0,
    )

    model_reg = PPO('MlpPolicy', train_env_reg,
                    **HYPERPARAMS_REGIME, verbose=0, seed=s)
    print(f'Training regime agent (seed={s}, zadj + online-calib obs)...')
    model_reg.learn(TOTAL_TIMESTEPS_REGIME, callback=eval_cb_reg, progress_bar=True)
    model_reg.save(f'{save_dir}/final')

print(f'Regime training complete for seeds {SEEDS}.')